# MiniDance Face — 人脸视频微调（Colab）

在已发布的 `minidance-base`（120M）上继续训练，让它学会生成连贯的人脸特写视频，同时保留风景能力。不增加任何参数。

- **在 Colab 网页里运行**（需要挂载 Google 网盘），运行时选 **A100**，最好开高内存（High-RAM）
- 直接打开：<https://colab.research.google.com/github/JuniverseLi/minidance/blob/face/notebooks/face_colab.ipynb>
- 一共两段，都能断点续跑（中途断开就重新运行全部单元格）：
  1. **数据**：从 OpenVid 挑单人面部特写 → 人脸居中裁切 → 编码成 latent，存到网盘 `MyDrive/minidance/face_latents_*/`。只需跑一次，耗时主要在下载视频
  2. **训练**：人脸 75% + 风景 25% 混合，人脸区域 loss 加权，8000 步。checkpoint 和样例 GIF 存到网盘 `MyDrive/minidance/runs/face/`
- 训练耗时以日志里的 `clips/s` 为准：总样本数 = `STEPS × BATCH`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!nproc

## 1. 挂载网盘，拉取代码

人脸相关代码在 GitHub 的 `face` 分支上（合并进 `main` 后把 `BRANCH` 改成 `'main'`）。

In [ ]:
REPO_URL = 'https://github.com/JuniverseLi/minidance.git'
BRANCH = 'face'
DRIVE_DIR = '/content/drive/MyDrive/minidance'

import os, glob, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError('网盘挂载失败：请在 Colab 网页版中打开本 notebook 运行') from e
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists('/content/minidance'):
    !git clone -q -b {BRANCH} {REPO_URL} /content/minidance
else:
    !git -C /content/minidance fetch -q origin
    !git -C /content/minidance reset -q --hard origin/{BRANCH}
%cd /content/minidance
assert os.path.exists('trainer/train_face.py'), f'{BRANCH} 分支上没有人脸代码'
!git log --oneline -1

In [ ]:
# Colab 自带 torch / opencv，只装缺的包
!pip install -q pylance diffusers safetensors "transformers>=4.51" huggingface_hub
import cv2
assert hasattr(cv2, 'FaceDetectorYN'), f'opencv {cv2.__version__} 太旧，没有 YuNet 人脸检测'

In [ ]:
# HF token（可选，只需读权限）：不设置也能跑，但下载 OpenVid 可能被限速
# 优先读 Colab 左侧「Secrets」里的 HF_TOKEN，没有的话手动粘贴，直接回车则跳过
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not token:
    from getpass import getpass
    token = getpass('HF token（回车跳过）: ').strip()
if token:
    os.environ['HF_TOKEN'] = token
print('HF_TOKEN set' if token else '匿名下载')

## 2. 数据（只需跑一次，结果在网盘）

### 2.1 元数据与筛选

先拉 OpenVid 的全部元数据（约 94 万条，约 10 分钟，缓存到网盘），再用标题挑出“单人、特写、真人、镜头平稳”的候选片段，打印每一步筛掉多少条。

- `ALL_SOURCES = False`：只用 pexels / pixabay / mixkit 三个图库，授权最干净，和风景数据一致
- 编码后人脸片段太少（比如不足 5000 条）时改成 `True`，会加入其他来源的片段；结果存到另一个目录，不会和之前的混在一起

In [ ]:
ALL_SOURCES = False
TAG = 'all' if ALL_SOURCES else 'stock'
META_ALL = f'{DRIVE_DIR}/openvid_meta_all.parquet'
FACES = f'{DRIVE_DIR}/faces_{TAG}.parquet'
FACE_DRIVE = f'{DRIVE_DIR}/face_latents_tight_{TAG}'

if not os.path.exists(META_ALL):
    !python dataset/fetch_openvid_meta.py --out /content/openvid_meta
    !python -c "import pandas as pd, glob; pd.concat(pd.read_parquet(f) for f in sorted(glob.glob('/content/openvid_meta/*.parquet'))).to_parquet('{META_ALL}')"
else:
    print('已有', META_ALL)
FLAG = '--all_sources' if ALL_SOURCES else ''
!python dataset/filter_faces.py --meta {META_ALL} --out {FACES} {FLAG}

### 2.2 人脸裁切 + 编码

每条候选片段：取 33 帧（约 12 fps）→ 逐帧人脸检测，只留“有且只有一张清晰的脸、脸不漂出画面”的 → 围绕人脸裁一个固定的正方形（脸约占边长的 3/4，紧凑特写：256 分辨率下脸能分到更多 token）→ 用 Wan2.2 VAE 编码，同时存一份人脸 mask。

- 每 512 条候选打包成一个分片，已完成的分片自动跳过；日志里有保留率、速度和剩余时间（ETA）
- 保留率通常不高（很多标题写着特写的片段，其实脸很小或者有好几个人），这是正常的

In [ ]:
!python dataset/encode_faces.py --parquet {FACES} --out {FACE_DRIVE} --batch 16 --workers 8

In [ ]:
# 统计 + 抽 4 条解码出来看裁切效果（4 段左右排列）
import numpy as np, pandas as pd, torch
from IPython.display import Image, display
from safetensors.torch import load_file
from model.vae import WanVAE
from trainer.trainer_utils import save_gif

shards = sorted(glob.glob(f'{FACE_DRIVE}/shard_*.safetensors'))
n = sum(len(pd.read_parquet(s.replace('.safetensors', '.parquet'))) for s in shards)
print(f'{len(shards)} 个分片，{n:,} 条人脸片段')

vae = WanVAE(dtype=torch.bfloat16)
z = load_file(shards[0])['latents'][:4].cuda()
videos = [((vae.decode(z[i:i + 1])[0].permute(1, 2, 3, 0).float() + 1) * 127.5).clamp(0, 255).byte().cpu().numpy()
          for i in range(len(z))]
save_gif(np.stack(videos), '/content/face_preview.gif')
del vae
torch.cuda.empty_cache()
display(Image(filename='/content/face_preview.gif'))

## 3. 训练

### 3.1 准备数据和初始权重（每次新开运行时都要跑）

- 人脸 latent 从网盘复制到本地磁盘（直接读网盘太慢）
- 风景 latent 和中英短字幕从 HF 下载（约 9 GB，几分钟）
- 初始权重 `minidance-base.safetensors`（只做了预训练的版本，不是 SFT 版）从 HF 下载

In [ ]:
FACE_LOCAL = '/content/face_latents'
os.makedirs(FACE_LOCAL, exist_ok=True)
!cp -n {FACE_DRIVE}/shard_* {FACE_LOCAL}/
print(len(glob.glob(f'{FACE_LOCAL}/shard_*.safetensors')), 'face shards local')

if not os.path.exists('/content/data/short_captions.jsonl'):
    !hf download JuniverseLi/minidance-scenery --repo-type dataset --local-dir /content/data
print(len(glob.glob('/content/data/latents/shard_*.safetensors')), 'scenery shards local')

INIT = '/content/weights/minidance-base.safetensors'
if not os.path.exists(INIT):
    !hf download JuniverseLi/MiniDance minidance-base.safetensors --local-dir /content/weights
assert os.path.exists(INIT), INIT

### 3.2 开始训练

- `BATCH = 64`：A100 40GB 放得下（base 模型 batch 32 约 14 GB）。爆显存（CUDA out of memory）就改 48，同时把 `LR` 调到 `8e-5`
- `STEPS = 8000`：学习率按它做余弦衰减，中途续跑**不要改**
- `MINUTES`：到时间自动出样例、保存再退出，下次重新运行全部单元格会从断点继续
- 每 1000 步出一张样例 GIF（从左到右：3 条人脸提示词、1 条中文风景提示词、训练集里的 1 条人脸和 1 条风景）。**人脸清楚稳定、风景没变差**就可以提前停
- `SCENERY_RATIO`：风景在每个 batch 里的占比。风景明显退化就调到 0.35，人脸学得慢就调到 0.15

In [ ]:
RUN = 'face'
STEPS = 8000
BATCH = 64
LR = 1e-4
MINUTES = 300
SCENERY_RATIO = 0.25
OUT = f'{DRIVE_DIR}/runs/{RUN}'

!python trainer/train_face.py --data_dir {FACE_LOCAL} \
    --scenery_dir /content/data/latents --scenery_captions /content/data/short_captions.jsonl \
    --scenery_ratio {SCENERY_RATIO} --face_weight 2.0 \
    --init {INIT} --preset base --batch {BATCH} --lr {LR} --warmup 500 --steps {STEPS} \
    --max_minutes {MINUTES} --sample_every 1000 --save_every 1000 --log_every 50 \
    --compile --resume --out {OUT}

## 4. 看样例

In [ ]:
from IPython.display import Image, display
gifs = sorted(glob.glob(f'{OUT}/samples/*.gif'))
print(len(gifs), 'sample GIFs')
for g in gifs[-3:]:
    print(os.path.basename(g))
    display(Image(filename=g))

## 5. 导出权重并生成视频

把 EMA 权重导出成 `minidance-face.safetensors`（bf16，约 240 MB）存到网盘，再用固定种子生成几条 mp4。

In [ ]:
FINAL = f'{DRIVE_DIR}/minidance-face.safetensors'
!python scripts/export_weights.py {OUT}/ckpt.pt {FINAL}
!python scripts/sample.py --ckpt {FINAL} --steps 50 --seed 0 --out {DRIVE_DIR}/samples_face \
    --prompt "The video features a young woman with curly hair, speaking calmly to the camera." \
             "The video shows an elderly man with white hair, turning his head and smiling gently." \
             "The video features a middle-aged man wearing glasses, talking in an office." \
             "海上日落，小浪拍岸，金色余晖映在水面。"